# EarningsLens — Stage 8b: RAG Pipeline (FAISS + Gemma 4)

Stage 8b builds the retrieval-augmented Q&A system that lets users ask natural language
questions about any company’s guidance history.

Commitment sentences are embedded with `all-MiniLM-L6-v2` and stored in a **FAISS**
flat inner-product index (cosine similarity on L2-normalised vectors). At query time the
most relevant commitments are retrieved and passed to **`gemma3:12b`** (Google Gemma 3,
12B parameter variant) running locally via Ollama as grounding context.

### Outputs
| File | Description | Consumed by |
|---|---|---|
| `faiss.index` | FAISS flat cosine index | Stage 9, Stage 10 |
| `faiss_meta.parquet` | Per-document metadata aligned to index rows | Stage 9, Stage 10 |
| `rag_eval_results.parquet` | RAG evaluation scores against Lamini QA | Project eval |

## 0. Dependencies

## 0.1 Imports & Logging

In [1]:
import re
import logging
import gc
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import psutil
from tqdm import tqdm
from datasets import load_dataset
import ollama
from sentence_transformers import SentenceTransformer

try:
    import torch
    _torch_available = True
except ImportError:
    _torch_available = False

logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO,
)
log = logging.getLogger("earningslens.stage8b")

print("✓ Imports OK")
print(f"  faiss version  : {faiss.__version__}")

c:\Users\sidsu\Desktop\Sem 4\NLP\NLPvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Imports OK
  faiss version  : 1.13.2


## 1. Configuration

In [2]:
# ── Paths
INPUT_PATH     = Path("fls_2023_2025.parquet")
FAISS_INDEX    = Path("faiss.index")
FAISS_META     = Path("faiss_meta.parquet")

# ── Model selection
# gemma3:12b  → ~8 GB VRAM, fits RTX 4060 exactly, recommended
# gemma3:4b   → ~3.5 GB VRAM, fastest response, lower quality
# gemma3:27b  → ~16 GB, splits VRAM+RAM (use num_gpu below to tune)
OLLAMA_MODEL   = "gemma3:12b"
EMBED_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM      = 384

# ── Ollama GPU options tuned for RTX 4060 8 GB VRAM
# num_gpu: number of transformer layers loaded onto GPU.
#   gemma3:12b has 46 layers — setting 46 keeps everything in VRAM.
#   If you get CUDA OOM errors, lower to 40 (remaining layers use RAM).
#   For gemma3:27b, try num_gpu=20 to split evenly across VRAM+RAM.
OLLAMA_OPTIONS = {
    "temperature" : 0,
    "num_gpu"     : 46,   # all layers on GPU for gemma3:12b
    "num_ctx"     : 4096, # context window (tokens). Reduce to 2048 if OOM.
}

# ── RAG parameters
TOP_K          = 5

# ── System info
ram = psutil.virtual_memory()
print(f"Available RAM  : {ram.available / 1e9:.1f} GB / {ram.total / 1e9:.1f} GB total")
if _torch_available and torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_free = torch.cuda.mem_get_info(0)[0] / 1e9
    vram_total = gpu.total_memory / 1e9
    print(f"GPU            : {gpu.name}")
    print(f"VRAM           : {vram_free:.1f} GB free / {vram_total:.1f} GB total")
else:
    print("GPU            : not detected (CPU mode)")
print()
print("✓ Configuration set")
print(f"  Ollama model   : {OLLAMA_MODEL}")
print(f"  Embedding model: {EMBED_MODEL}")
print(f"  FAISS index    : {FAISS_INDEX}")
print(f"  FAISS metadata : {FAISS_META}")
print(f"  Top-K retrieval: {TOP_K}")
print(f"  Ollama options : {OLLAMA_OPTIONS}")

Available RAM  : 5.1 GB / 16.8 GB total
GPU            : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM           : 7.4 GB free / 8.6 GB total

✓ Configuration set
  Ollama model   : gemma3:12b
  Embedding model: sentence-transformers/all-MiniLM-L6-v2
  FAISS index    : faiss.index
  FAISS metadata : faiss_meta.parquet
  Top-K retrieval: 5
  Ollama options : {'temperature': 0, 'num_gpu': 46, 'num_ctx': 4096}


## 2. Load Data

In [3]:
fls = pd.read_parquet(INPUT_PATH)
log.info("Loaded %d FLS rows", len(fls))

print(f"FLS sentences   : {len(fls):,}")
print(f"Unique tickers  : {fls['ticker'].nunique():,}")

2026-05-07 08:26:22,389 [INFO] Loaded 92458 FLS rows


FLS sentences   : 92,458
Unique tickers  : 531


## 3. Build FAISS Vector Index

All Specific FLS commitment sentences (those with `slot_count > 0`) are embedded with
`all-MiniLM-L6-v2` and added to a **`faiss.IndexFlatIP`** (flat inner-product) index.
Because vectors are L2-normalised before insertion, inner product equals cosine similarity.

Metadata (ticker, year, quarter, metric, value, timeframe, status, hedge_score) is stored
in `faiss_meta.parquet`, whose row order is **exactly aligned** with the FAISS index so
that `index.search()` position `i` maps directly to `meta_df.iloc[i]`.

Both files are written to disk so this step only runs once. If they already exist the
index is loaded directly.

In [4]:
if FAISS_INDEX.exists() and FAISS_META.exists():
    # ── Load existing index
    log.info("Loading existing FAISS index from %s", FAISS_INDEX)
    faiss_index = faiss.read_index(str(FAISS_INDEX))
    meta_df     = pd.read_parquet(FAISS_META)
    log.info("FAISS index loaded: %d vectors", faiss_index.ntotal)
    print(f"✓ Existing FAISS index loaded")
    print(f"  Vectors in index : {faiss_index.ntotal:,}")
    print(f"  Metadata rows    : {len(meta_df):,}")

else:
    # ── Build from scratch
    docs_df = fls[fls["slot_count"] > 0].copy().reset_index(drop=True)
    log.info("Building FAISS index from %d documents...", len(docs_df))

    # Step 1 ─ embed all sentences
    embedder  = SentenceTransformer(EMBED_MODEL)
    sentences = docs_df["sentence"].tolist()

    log.info("Encoding %d sentences (batch_size=64)...", len(sentences))
    embeddings = embedder.encode(
        sentences,
        batch_size           = 64,
        show_progress_bar    = True,
        normalize_embeddings = True,   # L2-normalise → IP == cosine
        convert_to_numpy     = True,
    ).astype("float32")               # FAISS requires float32
    log.info("Embeddings shape: %s", embeddings.shape)

    # Step 2 ─ free model memory before building index
    del embedder
    gc.collect()
    log.info("Embedder freed")

    # Step 3 ─ build FAISS flat index
    # IndexFlatIP = exact brute-force inner product search
    # (cosine similarity because vectors are already L2-normalised)
    faiss_index = faiss.IndexFlatIP(EMBED_DIM)
    faiss_index.add(embeddings)        # single numpy call — no batching needed
    log.info("FAISS index built: %d vectors", faiss_index.ntotal)

    # Step 4 ─ save index to disk
    faiss.write_index(faiss_index, str(FAISS_INDEX))
    log.info("FAISS index written to %s", FAISS_INDEX)

    # Step 5 ─ build and save aligned metadata table
    meta_df = docs_df[[
        "commitment_id", "ticker", "company_name",
        "year", "quarter", "sentence",
        "metric", "value", "timeframe", "status", "hedge_score",
    ]].copy()
    # Coerce types for safe parquet serialisation
    meta_df["year"]        = meta_df["year"].astype(int)
    meta_df["quarter"]     = meta_df["quarter"].astype(int)
    meta_df["hedge_score"] = pd.to_numeric(meta_df["hedge_score"], errors="coerce").fillna(0.0)
    for col in ["metric", "value", "timeframe", "status", "company_name"]:
        meta_df[col] = meta_df[col].fillna("").astype(str)
    meta_df.to_parquet(FAISS_META, index=False)
    log.info("Metadata written to %s", FAISS_META)

    del embeddings
    gc.collect()

    print(f"✓ FAISS index built and saved")
    print(f"  Vectors in index : {faiss_index.ntotal:,}")
    print(f"  Index file       : {FAISS_INDEX}  ({FAISS_INDEX.stat().st_size / 1e6:.1f} MB)")
    print(f"  Metadata file    : {FAISS_META}")

2026-05-07 08:26:23,374 [INFO] Building FAISS index from 65836 documents...
2026-05-07 08:26:23,377 [INFO] No device provided, using cuda:0
2026-05-07 08:26:23,837 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 08:26:23,853 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-07 08:26:24,104 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 08:26:24,127 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-07 08:26:24,129 [INFO] Loading SentenceTransform

✓ FAISS index built and saved
  Vectors in index : 65,836
  Index file       : faiss.index  (101.1 MB)
  Metadata file    : faiss_meta.parquet


## 4. RAG Query Function

Takes a natural language question and an optional ticker filter, retrieves the `TOP_K`
most relevant commitments from the FAISS index, and passes them as grounding context to
`gemma3:12b`. The LLM is instructed to answer solely from the retrieved context.

**Ticker filtering** is handled as a post-retrieval step: we retrieve `TOP_K × 20`
candidates from FAISS (which has no metadata awareness), then filter by ticker and
keep the top `TOP_K`. This is standard practice for flat FAISS indexes.

In [5]:
# Gemma 4 responds well to XML-style tags for context separation.
# The instruction to cite sources matches its instruction-following strengths.
RAG_PROMPT_TEMPLATE = """You are a financial analyst assistant answering questions about
management commitments extracted from corporate earnings call transcripts.

Answer the question using ONLY the information in the <context> block below.
Do not use prior knowledge or make assumptions beyond what is stated.
If the context does not contain enough information to answer, say so clearly.
Be specific: cite metric names, exact values, timeframes, and quarter/year when available.
Keep your answer concise and factual.

<context>
{context}
</context>

Question: {question}

Answer:"""

In [6]:
# Loaded once here, reused across every query call.
# This is the only SentenceTransformer instance alive during query time.
_query_embedder = SentenceTransformer(EMBED_MODEL)
print("✓ Query embedder loaded")

2026-05-07 08:26:46,729 [INFO] No device provided, using cuda:0
2026-05-07 08:26:47,014 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 08:26:47,037 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-07 08:26:47,274 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 08:26:47,298 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-07 08:26:47,298 [INFO] Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-05-07 08:26:47,53

✓ Query embedder loaded


In [7]:
def rag_query(question: str, ticker: str = None, top_k: int = TOP_K) -> dict:
    """
    Retrieve the top_k most relevant FLS commitments and generate an answer
    using Gemma 4 via Ollama.

    Parameters
    ----------
    question : str   Natural language question.
    ticker   : str   Optional ticker filter (e.g. 'AAPL'). None = all companies.
    top_k    : int   Number of documents to pass to the LLM.

    Returns
    -------
    dict with keys: answer, retrieved_docs, retrieved_metadata
    """
    # ── 1. Embed query
    q_vec = _query_embedder.encode(
        [question],
        normalize_embeddings = True,
        convert_to_numpy     = True,
    ).astype("float32")  # shape (1, 384)

    # ── 2. FAISS search
    # Over-fetch when filtering by ticker so enough candidates survive the filter.
    fetch_k = top_k * 20 if ticker else top_k
    scores, indices = faiss_index.search(q_vec, fetch_k)
    indices = indices[0]
    scores  = scores[0]

    # ── 3. Fetch metadata
    valid_mask = indices >= 0
    indices    = indices[valid_mask]
    scores     = scores[valid_mask]
    batch_meta = meta_df.iloc[indices]

    # ── 4. Optional ticker filter
    if ticker:
        keep       = batch_meta["ticker"] == ticker
        batch_meta = batch_meta[keep].head(top_k)
        scores     = scores[keep.values][:top_k]
    else:
        batch_meta = batch_meta.head(top_k)
        scores     = scores[:top_k]

    retrieved_docs     = batch_meta["sentence"].tolist()
    retrieved_metadata = batch_meta.to_dict(orient="records")

    # ── 5. Build context string
    context_lines = []
    for i, (doc, meta, score) in enumerate(
        zip(retrieved_docs, retrieved_metadata, scores), 1
    ):
        context_lines.append(
            f"{i}. [{meta.get('ticker')} Q{meta.get('quarter')} {meta.get('year')} "
            f"| {meta.get('metric')} | {meta.get('value')} | {meta.get('timeframe')} "
            f"| status={meta.get('status')} | score={score:.3f}]\n   {doc}"
        )
    context = "\n".join(context_lines)

    # ── 6. Generate answer with gemma3:12b via Ollama
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)
    try:
        response = ollama.chat(
            model    = OLLAMA_MODEL,
            messages = [{"role": "user", "content": prompt}],
            options  = OLLAMA_OPTIONS,  # GPU layers, context size, temperature
        )
        answer = response["message"]["content"].strip()
    except Exception as e:
        answer = f"Error generating answer: {e}"

    return {
        "answer"            : answer,
        "retrieved_docs"    : retrieved_docs,
        "retrieved_metadata": retrieved_metadata,
    }

In [8]:
# Warm up Ollama: loads gemma3:12b weights into VRAM before the evaluation loop.
# First call can take 30-60s while weights transfer from disk to GPU.
# Subsequent calls are fast (~1-3s per query on RTX 4060).
log.info("Warming up Ollama model %s (loading into VRAM)...", OLLAMA_MODEL)
try:
    ollama.chat(
        model    = OLLAMA_MODEL,
        messages = [{"role": "user", "content": "Reply with one word: ready"}],
        options  = OLLAMA_OPTIONS,
    )
    print(f"✓ Ollama warm-up complete ({OLLAMA_MODEL} loaded into VRAM)")
except Exception as e:
    print(f"⚠ Ollama warm-up failed: {e}")
    print("  Make sure Ollama is running and the model is pulled:")
    print(f"  > ollama pull {OLLAMA_MODEL}")

2026-05-07 08:26:52,303 [INFO] Warming up Ollama model gemma3:12b (loading into VRAM)...
2026-05-07 08:27:00,609 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


✓ Ollama warm-up complete (gemma3:12b loaded into VRAM)


## 5. Smoke Test

A few representative queries confirm retrieval and generation are working before
the full evaluation run.

In [9]:
test_queries = [
    {"question": "What revenue guidance did Apple give for 2024?",          "ticker": "AAPL"},
    {"question": "What capital expenditure commitments did companies make?", "ticker": None},
    {"question": "Which companies committed to margin improvement in 2025?", "ticker": None},
]

In [12]:
for q in test_queries:
    ticker_label = f"[{q['ticker']}]" if q["ticker"] else "[all]"
    print(f"\n── Question: {q['question']} {ticker_label} ──")
    result = rag_query(q["question"], ticker=q["ticker"])
    print(f"Answer: {result['answer'][:400]}")
    print(f"Retrieved: {len(result['retrieved_docs'])} documents")
    for i, meta in enumerate(result["retrieved_metadata"], 1):
        print(f"  {i}. {meta.get('ticker')} Q{meta.get('quarter')} {meta.get('year')} "
              f"| {meta.get('metric')} | {meta.get('value')}")


── Question: What revenue guidance did Apple give for 2024? [AAPL] ──


Batches: 100%|██████████| 1/1 [00:00<00:00, 89.11it/s]
2026-05-07 08:30:08,043 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Answer: Apple expects iPad revenue to grow double-digits in Q2 2024.
Retrieved: 5 documents
  1. AAPL Q3 2023 | revenue | to decline by double digits year-over-year
  2. AAPL Q1 2023 | revenue | to decline double digits year-over-year
  3. AAPL Q4 2023 | revenue | to grow year-over-year on an absolute basis
  4. AAPL Q2 2024 | revenue | double-digits
  5. AAPL Q4 2023 | revenue performance | 

── Question: What capital expenditure commitments did companies make? [all] ──


Batches: 100%|██████████| 1/1 [00:00<00:00, 98.33it/s]
2026-05-07 08:30:27,451 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Answer: Here's a summary of the capital expenditure commitments mentioned in the provided text:

*   **GPN:** Expects capital spending to be around $670 million, or roughly 7% of revenue, in 2024. Previously, GPN expected capital spending to be around $630 million in 2023.
*   **EXC:** Has a $38 billion capital plan, expecting to fund it with $20 billion of internally generated cash flow, $12 billion of d
Retrieved: 5 documents
  1. LNT Q3 2024 |  | approximately 40%
  2. GPN Q3 2024 |  | $670 million, or roughly 7% of revenue
  3. GPN Q1 2023 | capital spending | around $630 million
  4. EXC Q4 2024 |  | $38 billion
  5. CCI Q3 2023 | discretionary capital expenditures | approximately $1.2 billion

── Question: Which companies committed to margin improvement in 2025? [all] ──


Batches: 100%|██████████| 1/1 [00:00<00:00, 129.73it/s]
2026-05-07 08:30:31,562 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Answer: ITW expects margin improvement in specialty in 2025. Autodesk is planning for operating margin improvement in fiscal 2025.
Retrieved: 5 documents
  1. HON Q4 2023 | Margin | 
  2. ITW Q4 2024 | margin | 
  3. WTW Q4 2023 |  | 
  4. ADSK Q3 2024 | operating margin | 
  5. LH Q4 2023 |  | 


## Stage 8b Summary

In [11]:
print("=" * 52)
print("  EarningsLens — Stage 8b Complete")
print("=" * 52)
print(f"\nCorpus indexed       : {len(fls):,} FLS sentences")
print(f"Vectors in index     : 65,836 (slot_count > 0)")
print(f"Unique tickers       : {fls['ticker'].nunique():,}")
print(f"Unique transcripts   : {fls['transcript_id'].nunique():,}")
print(f"\nVector store         : {FAISS_INDEX}")
print(f"Metadata             : {FAISS_META}")
print(f"Embedding model      : {EMBED_MODEL}")
print(f"Generation model     : {OLLAMA_MODEL}")
print(f"\nSample queries       : 3 (ticker-scoped, cross-corpus, thematic)")
print(f"\n→ Ready for Stage 9 — Agentic Layer")

  EarningsLens — Stage 8b Complete

Corpus indexed       : 92,458 FLS sentences
Vectors in index     : 65,836 (slot_count > 0)
Unique tickers       : 531
Unique transcripts   : 4,664

Vector store         : faiss.index
Metadata             : faiss_meta.parquet
Embedding model      : sentence-transformers/all-MiniLM-L6-v2
Generation model     : gemma3:12b

Sample queries       : 3 (ticker-scoped, cross-corpus, thematic)

→ Ready for Stage 9 — Agentic Layer


| Output | Location | Consumed by |
|---|---|---|
| `faiss.index` | Local disk | Stage 9 (agentic), Stage 10 (dashboard) |
| `faiss_meta.parquet` | Local disk | Stage 9 (agentic), Stage 10 (dashboard) |
| `rag_query()` | In-memory function | Stage 9, Stage 10 |

### RAG Pipeline
- **Vector store** : FAISS flat inner-product index (cosine similarity on L2-normalised vectors)
- **Corpus** : 65,836 FLS commitment sentences (slot_count > 0) across 531 tickers (2023–2025)
- **Embedding model** : `all-MiniLM-L6-v2` (384-dim)
- **Retrieval** : Top-5 chunks, optional ticker filter for company-scoped queries
- **Generation** : `gemma3:12b` via Ollama — answers grounded in retrieved context

### Sample Queries
- Ticker-scoped query (AAPL revenue 2024) → correct retrieval, grounded answer
- Cross-corpus query (CapEx commitments) → multi-company synthesis across GPN, EXC, CCI
- Thematic query (margin improvement 2025) → ITW, Autodesk correctly identified